<a href="https://colab.research.google.com/github/NiloyKumarKundu/Graph-Neural-Network/blob/main/5_LSTM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np

def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))   # squashes any z into (0, 1): a "valve"

# ----------------------------------------------------------------------
# LSTM weights. Fixed here for the demo; normally LEARNED by backprop.
# Every gate has the SAME form:  activation(W_x * x_t + W_h * h_{t-1} + b),
# differing ONLY in its own weights.
#   - f, i, o use sigmoid  -> valves in (0, 1)
#   - candidate c_tilde uses tanh -> a value to store, in (-1, 1)
# ----------------------------------------------------------------------
P = {
    # forget gate : how much of OLD memory c_{t-1} to KEEP
    "W_xf": 1.0, "W_hf": 1.0, "b_f":  0.0,
    # input  gate : how much of the NEW candidate to WRITE
    "W_xi": 0.5, "W_hi": 0.5, "b_i": -0.5,
    # output gate : how much of the memory to EXPOSE as h_t
    "W_xo": 1.0, "W_ho": 0.5, "b_o":  0.0,
    # candidate   : the new info proposed for writing
    "W_xc": 1.0, "W_hc": 1.0, "b_c":  0.0,
}

def lstm_step(x_t, h_prev, c_prev, P, verbose=False):
    # x_t    : this step's input              (scalar here)
    # h_prev : previous hidden state  h_{t-1}
    # c_prev : previous cell state    c_{t-1}  (the protected memory highway)
    f     = sigmoid(P["W_xf"]*x_t + P["W_hf"]*h_prev + P["b_f"])   # forget valve
    i     = sigmoid(P["W_xi"]*x_t + P["W_hi"]*h_prev + P["b_i"])   # input  valve
    o     = sigmoid(P["W_xo"]*x_t + P["W_ho"]*h_prev + P["b_o"])   # output valve
    c_til = np.tanh( P["W_xc"]*x_t + P["W_hc"]*h_prev + P["b_c"])  # candidate (-1..1)

    c = f * c_prev + i * c_til     # cell update: KEEP old (xf) + WRITE new (xi), ADDED
    h = o * np.tanh(c)             # hidden = output valve x squashed memory

    if verbose:
        print(f"  z_f -> f = {f:.3f}   z_i -> i = {i:.3f}   "
              f"z_o -> o = {o:.3f}   z_c -> c_til = {c_til:.3f}")
        print(f"  c_t = f*c_prev + i*c_til = {f:.3f}*{c_prev:.3f} + "
              f"{i:.3f}*{c_til:.3f} = {c:.3f}")
        print(f"  h_t = o*tanh(c_t) = {o:.3f}*{np.tanh(c):.3f} = {h:.3f}")
    return h, c

# ===== PART 1 : single step, reproduce the by-hand numbers =====
print("PART 1 - single LSTM step  (c_prev=0.8, h_prev=0.5, x=0.60)")
lstm_step(0.60, h_prev=0.5, c_prev=0.8, P=P, verbose=True)

# ===== PART 2 : run the SAME cell across the full sequence =====
print("\nPART 2 - LSTM over the sequence [0.64, 0.60, 0.55, 0.48]")
x_seq = [0.64, 0.60, 0.55, 0.48]
h, c = 0.0, 0.0                       # h0 and c0 both start at 0
print(f"start:  h0 = {h:.3f}   c0 = {c:.3f}")
for t, x_t in enumerate(x_seq, start=1):
    h, c = lstm_step(x_t, h, c, P)    # feed previous (h, c) back in -> recurrence
    print(f"t={t}:  x={x_t:.2f}  ->  c{t} = {c:.3f}   h{t} = {h:.3f}")

PART 1 - single LSTM step  (c_prev=0.8, h_prev=0.5, x=0.60)
  z_f -> f = 0.750   z_i -> i = 0.512   z_o -> o = 0.701   z_c -> c_til = 0.800
  c_t = f*c_prev + i*c_til = 0.750*0.800 + 0.512*0.800 = 1.010
  h_t = o*tanh(c_t) = 0.701*0.766 = 0.537

PART 2 - LSTM over the sequence [0.64, 0.60, 0.55, 0.48]
start:  h0 = 0.000   c0 = 0.000
t=1:  x=0.64  ->  c1 = 0.257   h1 = 0.165
t=2:  x=0.60  ->  c2 = 0.478   h2 = 0.296
t=3:  x=0.55  ->  c3 = 0.666   h3 = 0.389
t=4:  x=0.48  ->  c4 = 0.808   h4 = 0.443
